In [1]:
import os, random, io, warnings
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix
)

warnings.filterwarnings('ignore')

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 60)
print(f"🚀 FREQUENCY DOMAIN DEEPFAKE DETECTION")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Random Seed: {SEED}")
print("=" * 60)


DATASET_ROOT = Path("/kaggle/input")  


IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


🚀 FREQUENCY DOMAIN DEEPFAKE DETECTION
Device: cuda
Random Seed: 42


In [4]:
CONFIG = {

    'target_size': 256,      
    'resize_size': 280,      
    'batch_size': 32,        
    'num_workers': 2,
    

    'use_phase': True,       
    'focus_high_freq': True, 
    'high_freq_threshold': 0.25,  
    

    'quality_aug_prob': 0.7,    
    'freq_mask_prob': 0.3,       
    'spatial_aug_prob': 0.5,     
    

    'epochs': 15,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'use_focal_loss': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'use_pretrained': True,      
    

    'val_freq': 1,              
    'save_best_only': True,
    'early_stopping_patience': 5,
    'monitor_quality_bias': True,
    

    'quality_balance_ratio': 0.5,  
    'balance_classes': True,
}

SAVE_PATH = "/kaggle/working/frequency_debiased_model_v3.pth"

print("\n📋 Configuration:")
for key, val in CONFIG.items():
    print(f"  {key}: {val}")
print("=" * 60)


📋 Configuration:
  target_size: 256
  resize_size: 280
  batch_size: 32
  num_workers: 2
  use_phase: True
  focus_high_freq: True
  high_freq_threshold: 0.25
  quality_aug_prob: 0.7
  freq_mask_prob: 0.3
  spatial_aug_prob: 0.5
  epochs: 15
  lr: 0.0001
  weight_decay: 0.0001
  use_focal_loss: True
  focal_alpha: 0.25
  focal_gamma: 2.0
  use_pretrained: True
  val_freq: 1
  save_best_only: True
  early_stopping_patience: 5
  monitor_quality_bias: True
  quality_balance_ratio: 0.5
  balance_classes: True


In [5]:

def rglob_images(p: Path) -> List[Path]:
    """Recursively find all images in directory."""
    if not p.exists():
        return []
    return [x for x in p.rglob("*") if x.suffix.lower() in IMG_EXTS]


def find_dir_contains(name: str) -> Path:
    """Find directory containing specific name in path."""
    name = name.lower()
    for p in DATASET_ROOT.rglob("*"):
        if p.is_dir() and name in str(p).lower():
            return p
    raise FileNotFoundError(f"Could not find folder containing: {name}")



In [20]:

def build_quality_aware_index():
    """
    Build dataset index with quality labels.
    """
    print("\n🔍 Building dataset index...")
    
    index = {
        "real_hq": [],
        "real_lq": [],
        "fake_gan": [],
        "fake_diff": []
    }
    

    try:
        stylegan_real = rglob_images(find_dir_contains("train/real"))
        stylegan_fake = rglob_images(find_dir_contains("train/fake"))
        index["real_hq"].extend(stylegan_real)
        index["fake_gan"].extend(stylegan_fake)
        print(f"  ✓ StyleGAN - Real: {len(stylegan_real)}, Fake: {len(stylegan_fake)}")
    except FileNotFoundError:
        print("  ⚠ StyleGAN dataset not found")
    

    try:
        celeba = rglob_images(find_dir_contains("celeba"))
        index["real_hq"].extend(celeba)
        print(f"  ✓ CelebA - Real: {len(celeba)}")
    except FileNotFoundError:
        print("  ⚠ CelebA dataset not found")
    

    try:

        wish_root = None
        possible_paths = [
            "realvsfake-81k",
            "realvsfake",
            "RealVsFake",
            "wish"
        ]
        
        for path_name in possible_paths:
            try:
                wish_root = find_dir_contains(path_name)
                break
            except FileNotFoundError:
                continue
        
        if wish_root is None:
            raise FileNotFoundError("Wish dataset not found")
        


        wish_real_path = None
        wish_fake_path = None
        

        for p in wish_root.rglob("*"):
            if p.is_dir() and p.name == "Real":
                wish_real_path = p
            if p.is_dir() and p.name == "Fake":
                wish_fake_path = p
        
        if wish_real_path:
            wish_real = rglob_images(wish_real_path)
            index["real_lq"].extend(wish_real)
            print(f"  ✓ Wish - Real: {len(wish_real)}")
        
        if wish_fake_path:
            wish_fake = rglob_images(wish_fake_path)
            index["fake_gan"].extend(wish_fake)
            print(f"  ✓ Wish - Fake: {len(wish_fake)}")
        
        if not wish_real_path and not wish_fake_path:
            print("  ⚠ Wish Real/Fake folders not found in structure")
            
    except FileNotFoundError:
        print("  ⚠ Wish dataset not found")
    

    try:
        diffusion = rglob_images(find_dir_contains("syntheticeye"))
        index["fake_diff"].extend(diffusion)
        print(f"  ✓ Diffusion - Fake: {len(diffusion)}")
    except FileNotFoundError:
        print("  ⚠ Diffusion dataset not found")
    
    print(f"\n📊 Index Summary:")
    print(f"  Real HQ:    {len(index['real_hq']):,}")
    print(f"  Real LQ:    {len(index['real_lq']):,}")
    print(f"  Fake GAN:   {len(index['fake_gan']):,}")
    print(f"  Fake Diff:  {len(index['fake_diff']):,}")
    print(f"  Total:      {sum(len(v) for v in index.values()):,}")
    
    return index

In [9]:

def create_quality_balanced_splits(
    index: Dict[str, List[Path]], 
    train_ratio: float = 0.7,
    val_ratio: float = 0.1,
    quality_balance: float = 0.5
) -> Dict[str, List[Tuple[Path, int, str]]]:
    """
    Create train/val/test splits with balanced quality distribution.
    
    Args:
        index: Dataset index from build_quality_aware_index()
        train_ratio: Proportion for training
        val_ratio: Proportion for validation  
        quality_balance: Ratio of HQ to total (0.5 = equal HQ and LQ)
    
    Returns:
        Dict with 'train', 'val', 'test' keys containing list of tuples:
        (path, label, quality_tag) where quality_tag in ['hq', 'lq']
    """
    print(f"\n⚖️ Creating quality-balanced splits...")
    print(f"  Quality balance ratio: {quality_balance:.2f}")
    print(f"  Train/Val/Test: {train_ratio:.1%}/{val_ratio:.1%}/{1-train_ratio-val_ratio:.1%}")
    

    real_hq = [(p, 0, 'hq') for p in index['real_hq']]
    real_lq = [(p, 0, 'lq') for p in index['real_lq']]
    fake_gan = [(p, 1, 'mq') for p in index['fake_gan']]  
    fake_diff = [(p, 1, 'hq') for p in index['fake_diff']]
    

    random.shuffle(real_hq)
    random.shuffle(real_lq)
    random.shuffle(fake_gan)
    random.shuffle(fake_diff)
    

    n_real_samples = min(len(real_hq), len(real_lq)) * 2
    n_real_hq = int(n_real_samples * quality_balance)
    n_real_lq = n_real_samples - n_real_hq
    
    real_balanced = real_hq[:n_real_hq] + real_lq[:n_real_lq]
    random.shuffle(real_balanced)
    

    n_fake_samples = n_real_samples  
    n_fake_hq = min(len(fake_diff), int(n_fake_samples * 0.4))
    n_fake_mq = n_fake_samples - n_fake_hq
    
    fake_balanced = fake_diff[:n_fake_hq] + fake_gan[:n_fake_mq]
    random.shuffle(fake_balanced)
    
    print(f"\n  Balanced dataset sizes:")
    print(f"    Real: {len(real_balanced):,} ({n_real_hq:,} HQ + {n_real_lq:,} LQ)")
    print(f"    Fake: {len(fake_balanced):,} ({n_fake_hq:,} HQ + {n_fake_mq:,} MQ)")
    

    def split_list(lst, train_r, val_r):
        n = len(lst)
        t = int(train_r * n)
        v = int(val_r * n)
        return {
            'train': lst[:t],
            'val': lst[t:t+v],
            'test': lst[t+v:]
        }
    
    real_splits = split_list(real_balanced, train_ratio, val_ratio)
    fake_splits = split_list(fake_balanced, train_ratio, val_ratio)
    

    splits = {}
    for split_name in ['train', 'val', 'test']:
        combined = real_splits[split_name] + fake_splits[split_name]
        random.shuffle(combined)
        splits[split_name] = combined
        

        n_real = sum(1 for _, label, _ in combined if label == 0)
        n_fake = len(combined) - n_real
        n_hq = sum(1 for _, _, q in combined if q == 'hq')
        print(f"\n  {split_name.upper()}: {len(combined):,} samples")
        print(f"    Real/Fake: {n_real:,}/{n_fake:,}")
        print(f"    HQ ratio: {n_hq/len(combined):.1%}")
    
    return splits


In [10]:

class QualityInvariantAugmentation:
    """
    Apply quality degradation to BOTH real and fake images.
    This prevents the model from using quality as a discriminative feature.
    """
    
    def __init__(self, apply_prob=0.7):
        self.apply_prob = apply_prob
    
    def __call__(self, img: Image.Image) -> Image.Image:
        """Apply random quality degradation."""
        if random.random() > self.apply_prob:
            return img
        

        aug_type = random.choice([
            'jpeg', 'resize', 'blur', 'noise', 'combined'
        ])
        
        if aug_type == 'jpeg':
            img = self._jpeg_compress(img)
        elif aug_type == 'resize':
            img = self._resize_artifact(img)
        elif aug_type == 'blur':
            img = self._blur(img)
        elif aug_type == 'noise':
            img = self._add_noise(img)
        else:

            if random.random() < 0.6:
                img = self._jpeg_compress(img)
            if random.random() < 0.4:
                img = self._resize_artifact(img)
            if random.random() < 0.3:
                img = self._blur(img)
        
        return img
    
    def _jpeg_compress(self, img):
        """JPEG compression artifacts."""
        buf = io.BytesIO()
        quality = random.randint(55, 95)
        img.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        return Image.open(buf).convert('RGB')
    
    def _resize_artifact(self, img):
        """Downsampling + upsampling artifacts."""
        w, h = img.size
        scale = random.uniform(0.6, 0.9)
        small = img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)
        return small.resize((w, h), Image.BICUBIC)
    
    def _blur(self, img):
        """Gaussian blur."""
        radius = random.uniform(0.3, 1.5)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))
    
    def _add_noise(self, img):
        """Add Gaussian noise."""
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(3, 12), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)



In [11]:
def improved_fft_transform(
    img_tensor: torch.Tensor,
    use_phase: bool = True,
    focus_high_freq: bool = True,
    high_freq_threshold: float = 0.25
) -> torch.Tensor:
    """
    Enhanced FFT transform with multi-channel processing and phase info.
    
    Args:
        img_tensor: [C, H, W] RGB image tensor (0-1 range)
        use_phase: Include phase information
        focus_high_freq: Apply high-pass filter
        high_freq_threshold: Radial threshold for high-pass (0-1)
    
    Returns:
        FFT features: [C*2, H, W] if use_phase else [C, H, W]
    """
    C, H, W = img_tensor.shape
    features = []
    

    for c in range(C):

        fft = torch.fft.fftshift(torch.fft.fft2(img_tensor[c]))
        

        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)  
        
        features.append(magnitude)
        

        if use_phase:
            phase = torch.angle(fft)
            features.append(phase)
    

    fft_features = torch.stack(features, dim=0)  
    

    if focus_high_freq:
        cy, cx = H // 2, W // 2
        

        y_coords = torch.arange(H, dtype=torch.float32) - cy
        x_coords = torch.arange(W, dtype=torch.float32) - cx
        Y, X = torch.meshgrid(y_coords, x_coords, indexing='ij')
        radius = torch.sqrt(Y**2 + X**2)
        max_radius = radius.max()
        

        mask = (radius > high_freq_threshold * max_radius).float()
        mask = mask.unsqueeze(0)  
        
        fft_features = fft_features * mask
    

    fft_features = fft_features / (fft_features.std() + 1e-8)
    fft_features = torch.clamp(fft_features, -10, 10)
    
    return fft_features



In [12]:
def frequency_mask_augmentation(x: torch.Tensor, mask_prob: float = 0.3) -> torch.Tensor:
    """
    Random frequency masking for regularization.
    """
    if random.random() > mask_prob:
        return x
    
    C, H, W = x.shape
    

    radius = random.randint(5, 30)
    cy, cx = H // 2, W // 2
    
    x_copy = x.clone()
    x_copy[:, cy-radius:cy+radius, cx-radius:cx+radius] = 0
    
    return x_copy


In [14]:
class FrequencyDeepfakeDataset(Dataset):
    """
    Dataset for frequency-domain deepfake detection.
    Applies quality-invariant augmentation and FFT transform.
    """
    
    def __init__(
        self, 
        samples: List[Tuple[Path, int, str]],
        config: dict,
        is_train: bool = True
    ):
        """
        Args:
            samples: List of (path, label, quality_tag) tuples
            config: Configuration dictionary
            is_train: Whether this is training set
        """
        self.samples = samples
        self.config = config
        self.is_train = is_train
        

        if is_train:
            self.quality_aug = QualityInvariantAugmentation(
                apply_prob=config['quality_aug_prob']
            )
        else:
            self.quality_aug = None
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label, quality_tag = self.samples[idx]
        

        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading {path}: {e}")

            return self.__getitem__(random.randint(0, len(self) - 1))
        

        img = img.resize(
            (self.config['resize_size'], self.config['resize_size']), 
            Image.BICUBIC
        )
        
        left = (self.config['resize_size'] - self.config['target_size']) // 2
        img = img.crop((
            left, left, 
            left + self.config['target_size'], 
            left + self.config['target_size']
        ))
        

        if self.is_train and self.quality_aug:
            img = self.quality_aug(img)
        

        img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        

        fft_features = improved_fft_transform(
            img_tensor,
            use_phase=self.config['use_phase'],
            focus_high_freq=self.config['focus_high_freq'],
            high_freq_threshold=self.config['high_freq_threshold']
        )
        

        if self.is_train:
            fft_features = frequency_mask_augmentation(
                fft_features, 
                mask_prob=self.config['freq_mask_prob']
            )
        
        return fft_features, torch.tensor(label, dtype=torch.long)



In [15]:
class FrequencyDeepfakeDetector(nn.Module):
    """
    Frequency-domain deepfake detection model.
    Uses ResNet18 backbone with frequency-aware modifications.
    """
    
    def __init__(
        self, 
        input_channels: int = 6,  
        use_pretrained: bool = True
    ):
        super().__init__()
        

        if use_pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
            self.backbone = resnet18(weights=weights)
        else:
            self.backbone = resnet18(weights=None)
        

        if input_channels != 3:
            self.backbone.conv1 = nn.Conv2d(
                input_channels, 64,
                kernel_size=7, stride=2, padding=3, bias=False
            )

            nn.init.kaiming_normal_(
                self.backbone.conv1.weight, 
                mode='fan_out', 
                nonlinearity='relu'
            )
        

        self.backbone.fc = nn.Identity()
        

        self.freq_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 512),
            nn.Sigmoid()
        )
        

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    
    def forward(self, x):

        features = self.backbone(x)  
        

        attention_weights = self.freq_attention(features.view(features.size(0), 512, 1, 1))
        features = features * attention_weights
        

        return self.classifier(features)




In [16]:

class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance and hard examples.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')
    
    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()



In [17]:

def evaluate_model(
    model: nn.Module, 
    loader: DataLoader, 
    device: str,
    split_name: str = "VAL"
) -> Dict[str, float]:
    """
    Evaluate model on validation/test set.
    
    Returns:
        Dictionary with accuracy, precision, recall, F1
    """
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):
            x, y = x.to(device), y.to(device)
            
            outputs = model(x)
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  
    

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    

    cm = confusion_matrix(all_labels, all_preds)
    
    metrics = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'confusion_matrix': cm
    }
    
    return metrics


def print_metrics(metrics: Dict, split_name: str = "VAL"):
    """Pretty print metrics."""
    print(f"\n{'='*60}")
    print(f"{split_name} METRICS:")
    print(f"{'='*60}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    
    if 'confusion_matrix' in metrics:
        cm = metrics['confusion_matrix']
        print(f"\n  Confusion Matrix:")
        print(f"                Predicted")
        print(f"              Real   Fake")
        print(f"  Actual Real  {cm[0,0]:5d}  {cm[0,1]:5d}")
        print(f"        Fake  {cm[1,0]:5d}  {cm[1,1]:5d}")
    
    print(f"{'='*60}")



In [18]:

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: dict,
    device: str,
    save_path: str
):
    """
    Train the frequency deepfake detection model.
    """
    print(f"\n{'='*60}")
    print("🚀 STARTING TRAINING")
    print(f"{'='*60}")
    

    if config['use_focal_loss']:
        criterion = FocalLoss(
            alpha=config['focal_alpha'],
            gamma=config['focal_gamma']
        )
        print(f"Using Focal Loss (alpha={config['focal_alpha']}, gamma={config['focal_gamma']})")
    else:
        criterion = nn.CrossEntropyLoss()
        print("Using Cross-Entropy Loss")
    

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=config['epochs']
    )
    

    best_val_f1 = 0.0
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch + 1}/{config['epochs']}")
        print(f"{'='*60}")
        

        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
        for batch_idx, (x, y) in enumerate(progress_bar):
            x, y = x.to(device), y.to(device)
            

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            

            loss.backward()
            optimizer.step()
            

            train_loss += loss.item()
            preds = outputs.argmax(1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)
            

            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.0 * train_correct / train_total:.2f}%'
            })
        

        avg_train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        
        print(f"\n📊 Training Results:")
        print(f"  Loss: {avg_train_loss:.4f}")
        print(f"  Accuracy: {train_acc:.4f}")
        print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
        

        if (epoch + 1) % config['val_freq'] == 0:
            val_metrics = evaluate_model(model, val_loader, device, "VALIDATION")
            print_metrics(val_metrics, "VALIDATION")
            

            if config['save_best_only']:
                if val_metrics['f1'] > best_val_f1:
                    best_val_f1 = val_metrics['f1']
                    patience_counter = 0
                    
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_f1': best_val_f1,
                        'val_metrics': val_metrics,
                        'config': config
                    }, save_path)
                    
                    print(f"\n✅ Best model saved! (F1: {best_val_f1:.4f})")
                else:
                    patience_counter += 1
                    print(f"\n⏳ No improvement. Patience: {patience_counter}/{config['early_stopping_patience']}")
            else:

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_metrics': val_metrics,
                    'config': config
                }, save_path.replace('.pth', f'_epoch{epoch+1}.pth'))
        

        if patience_counter >= config['early_stopping_patience']:
            print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
            break
        

        scheduler.step()
    
    print(f"\n{'='*60}")
    print("✅ TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Best Validation F1: {best_val_f1:.4f}")
    print(f"Model saved to: {save_path}")
    
    return model


In [ ]:

if __name__ == "__main__":

    index = build_quality_aware_index()
    

    splits = create_quality_balanced_splits(
        index,
        train_ratio=0.7,
        val_ratio=0.1,
        quality_balance=CONFIG['quality_balance_ratio']
    )
    

    print(f"\n{'='*60}")
    print("📦 Creating Datasets...")
    print(f"{'='*60}")
    
    train_dataset = FrequencyDeepfakeDataset(
        splits['train'], 
        CONFIG, 
        is_train=True
    )
    val_dataset = FrequencyDeepfakeDataset(
        splits['val'], 
        CONFIG, 
        is_train=False
    )
    test_dataset = FrequencyDeepfakeDataset(
        splits['test'], 
        CONFIG, 
        is_train=False
    )
    
    print(f"  Train: {len(train_dataset):,} samples")
    print(f"  Val:   {len(val_dataset):,} samples")
    print(f"  Test:  {len(test_dataset):,} samples")
    

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    

    print(f"\n{'='*60}")
    print("🏗️ Building Model...")
    print(f"{'='*60}")
    
    input_channels = 6 if CONFIG['use_phase'] else 3
    model = FrequencyDeepfakeDetector(
        input_channels=input_channels,
        use_pretrained=CONFIG['use_pretrained']
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"  Input Channels: {input_channels}")
    print(f"  Total Parameters: {total_params:,}")
    print(f"  Trainable Parameters: {trainable_params:,}")
    

    model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=CONFIG,
        device=DEVICE,
        save_path=SAVE_PATH
    )
    

    print(f"\n{'='*60}")
    print("🎯 FINAL TEST EVALUATION")
    print(f"{'='*60}")
    

    checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_metrics = evaluate_model(model, test_loader, DEVICE, "TEST")
    print_metrics(test_metrics, "FINAL TEST")
    
    print(f"\n{'='*60}")
    print("🎉 ALL DONE!")
    print(f"{'='*60}")
    print(f"Model saved at: {SAVE_PATH}")
    print(f"Test F1 Score: {test_metrics['f1']:.4f}")
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")



🔍 Building dataset index...
  ✓ StyleGAN - Real: 50000, Fake: 50000
  ✓ CelebA - Real: 202599
  ✓ Wish - Real: 81000
  ✓ Wish - Fake: 80972
  ✓ Diffusion - Fake: 33578

📊 Index Summary:
  Real HQ:    252,599
  Real LQ:    81,000
  Fake GAN:   130,972
  Fake Diff:  33,578
  Total:      498,149

⚖️ Creating quality-balanced splits...
  Quality balance ratio: 0.50
  Train/Val/Test: 70.0%/10.0%/20.0%

  Balanced dataset sizes:
    Real: 162,000 (81,000 HQ + 81,000 LQ)
    Fake: 162,000 (33,578 HQ + 128,422 MQ)

  TRAIN: 226,800 samples
    Real/Fake: 113,400/113,400
    HQ ratio: 35.4%

  VAL: 32,400 samples
    Real/Fake: 16,200/16,200
    HQ ratio: 35.6%

  TEST: 64,800 samples
    Real/Fake: 32,400/32,400
    HQ ratio: 35.1%

📦 Creating Datasets...
  Train: 226,800 samples
  Val:   32,400 samples
  Test:  64,800 samples

🏗️ Building Model...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 163MB/s] 


  Input Channels: 6
  Total Parameters: 11,449,474
  Trainable Parameters: 11,449,474

🚀 STARTING TRAINING
Using Focal Loss (alpha=0.25, gamma=2.0)

EPOCH 1/15


Training Epoch 1: 100%|██████████| 7088/7088 [42:39<00:00,  2.77it/s, loss=0.0172, acc=84.32%]



📊 Training Results:
  Loss: 0.0214
  Accuracy: 0.8432
  Learning Rate: 0.000100


...


VALIDATION METRICS:
  Accuracy:  0.9594
  Precision: 0.9753
  Recall:    0.9426
  F1 Score:  0.9587

  Confusion Matrix:
                Predicted
              Real   Fake
  Actual Real  15813    387
        Fake    930  15270

⏳ No improvement. Patience: 1/5

EPOCH 8/15


Training Epoch 8: 100%|██████████| 7088/7088 [35:27<00:00,  3.33it/s, loss=0.0114, acc=94.98%]



📊 Training Results:
  Loss: 0.0078
  Accuracy: 0.9498
  Learning Rate: 0.000055



VALIDATION METRICS:
  Accuracy:  0.9619
  Precision: 0.9771
  Recall:    0.9459
  F1 Score:  0.9613

  Confusion Matrix:
                Predicted
              Real   Fake
  Actual Real  15841    359
        Fake    876  15324

⏳ No improvement. Patience: 2/5

EPOCH 9/15


Training Epoch 9:  59%|█████▉    | 4174/7088 [21:00<13:38,  3.56it/s, loss=0.0085, acc=95.41%]

In [24]:
print("plug")

plug
